In [22]:
import pandas as pd
import os
from pathlib import Path

In [23]:
price_d1 = pd.read_csv(Path("data") / "prices_round_0_day_-1.csv", sep=";")
price_d1[price_d1["product"] == "EMERALDS"]

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
1,-1,0,EMERALDS,9992,14,9990,29,NaN,NaN,10008,14,10010,29,NaN,NaN,10000.0,0.0
2,-1,100,EMERALDS,9992,11,9990,22,NaN,NaN,10008,11,10010,22,NaN,NaN,10000.0,0.0
4,-1,200,EMERALDS,9992,15,9990,20,NaN,NaN,10008,15,10010,20,NaN,NaN,10000.0,0.0
7,-1,300,EMERALDS,9992,11,9990,29,NaN,NaN,10008,11,10010,29,NaN,NaN,10000.0,0.0
9,-1,400,EMERALDS,9992,12,9990,25,NaN,NaN,10008,12,10010,25,NaN,NaN,10000.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19991,-1,999500,EMERALDS,9992,14,9990,29,NaN,NaN,10008,14,10010,29,NaN,NaN,10000.0,0.0
19992,-1,999600,EMERALDS,9992,14,9990,28,NaN,NaN,10008,14,10010,28,NaN,NaN,10000.0,0.0
19995,-1,999700,EMERALDS,9992,12,9990,26,NaN,NaN,10008,12,10010,26,NaN,NaN,10000.0,0.0
19996,-1,999800,EMERALDS,9992,13,9990,20,NaN,NaN,10008,13,10010,20,NaN,NaN,10000.0,0.0


In [24]:
price_d1 = pd.read_csv("data/prices_round_0_day_-2.csv", sep=";")
price_d1.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-2,0,EMERALDS,9992,11,9990,25,NaN,NaN,10008,11,10010,25,NaN,NaN,10000.0,0.0
1,-2,0,TOMATOES,4993,7,4992,17,NaN,NaN,5007,7,5008,17,NaN,NaN,5000.0,0.0
2,-2,100,TOMATOES,4998,5,4993,7,4992.0,16.0,5007,7,5008,16,NaN,NaN,5002.5,0.0
3,-2,100,EMERALDS,9992,15,9990,20,NaN,NaN,10008,15,10010,20,NaN,NaN,10000.0,0.0
4,-2,200,TOMATOES,4994,6,4993,20,NaN,NaN,5008,6,5009,20,NaN,NaN,5001.0,0.0


In [25]:
trade_d1 = pd.read_csv("data/trades_round_0_day_-1.csv", sep=";")
trade_d1.sort_values("timestamp", inplace=True)
trade_d1.head()

,timestamp,buyer,seller,symbol,currency,price,quantity
0,3200,NaN,NaN,EMERALDS,XIRECS,9992.0,8
1,3400,NaN,NaN,TOMATOES,XIRECS,5009.0,2
2,5000,NaN,NaN,EMERALDS,XIRECS,9992.0,7
3,7000,NaN,NaN,TOMATOES,XIRECS,5010.0,4
4,9600,NaN,NaN,TOMATOES,XIRECS,4999.0,5


In [27]:
em = trade_d1[trade_d1["symbol"] == "EMERALDS"]
tom = trade_d1[trade_d1["symbol"] == "TOMATOES"]

In [28]:
trade_d2 = pd.read_csv("data/trades_round_0_day_-2.csv", sep=";")
trade_d2.sort_values("timestamp", inplace=True)
trade_d2.head()

,timestamp,buyer,seller,symbol,currency,price,quantity
0,900,NaN,NaN,TOMATOES,XIRECS,5008.0,2
1,1700,NaN,NaN,TOMATOES,XIRECS,5006.0,3
2,4000,NaN,NaN,EMERALDS,XIRECS,10008.0,7
3,4100,NaN,NaN,TOMATOES,XIRECS,5002.0,3
4,5200,NaN,NaN,EMERALDS,XIRECS,9992.0,5


In [29]:

# ── Merge prices + trades, classify level (1/2/3) and side (bid/ask) ─────────

def classify_trade(row):
    """Return (level, side) by matching trade price against order-book levels."""
    p = row["trade_price"]
    for lvl in [1, 2, 3]:
        if row[f"bid_price_{lvl}"] == p:
            return lvl, "bid"
        if row[f"ask_price_{lvl}"] == p:
            return lvl, "ask"
    # No exact level match — infer side from position relative to mid
    side = "bid" if p <= row["mid_price"] else "ask"
    return None, side


def build_merged(price_file, trade_file):
    prices = pd.read_csv(price_file, sep=";").sort_values("timestamp")
    trades = pd.read_csv(trade_file, sep=";").sort_values("timestamp")

    # Join each trade to the most recent price snapshot for that product
    merged = pd.merge_asof(
        trades.rename(columns={"symbol": "product", "price": "trade_price"}),
        prices,
        on="timestamp",
        by="product",
        direction="backward",
    )

    levels, sides = zip(*merged.apply(classify_trade, axis=1))
    merged["level"] = levels
    merged["side"]  = sides

    return merged[[
        "timestamp", "product", "trade_price", "quantity",
        "mid_price", "side", "level",
        "bid_price_1", "bid_price_2", "bid_price_3",
        "ask_price_1", "ask_price_2", "ask_price_3",
    ]].rename(columns={"trade_price": "price"})


merged_d1 = build_merged("data/prices_round_0_day_-1.csv", "data/trades_round_0_day_-1.csv")
merged_d2 = build_merged("data/prices_round_0_day_-2.csv", "data/trades_round_0_day_-2.csv")

print(f"Day -1: {len(merged_d1)} trades")
print(f"Day -2: {len(merged_d2)} trades")
merged_d1.head(10)


Day -1: 631 trades
Day -2: 588 trades


,timestamp,product,price,quantity,mid_price,side,level,bid_price_1,bid_price_2,bid_price_3,ask_price_1,ask_price_2,ask_price_3
0,3200,EMERALDS,9992.0,8,10000.0,bid,1,9992,9990,NaN,10008,10010,NaN
1,3400,TOMATOES,5009.0,2,5002.5,ask,1,4996,4994,NaN,5009,5011,NaN
2,5000,EMERALDS,9992.0,7,9996.0,bid,1,9992,9990,NaN,10000,10008,10010.0
3,7000,TOMATOES,5010.0,4,5003.5,ask,1,4997,4996,NaN,5010,5012,NaN
4,9600,TOMATOES,4999.0,5,5006.0,bid,1,4999,4998,NaN,5013,5014,NaN
5,9900,TOMATOES,5000.0,4,5006.5,bid,1,5000,4999,NaN,5013,5015,NaN
6,16400,TOMATOES,4996.0,2,5003.0,bid,1,4996,4995,NaN,5010,5011,NaN
7,17900,TOMATOES,5009.0,2,5002.0,ask,1,4995,4994,NaN,5009,5010,NaN
8,22200,TOMATOES,5005.0,3,4998.0,ask,1,4991,4990,NaN,5005,5006,NaN
9,23500,TOMATOES,4989.0,3,4995.5,bid,1,4989,4988,NaN,5002,5004,NaN
